In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

/Users/oguz/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv("train.csv")

df.head()

,symbol,event_time,bid_price,bid_qty,ask_price,ask_qty
0,ETHUSDT,1775653725357699,2265.77,19.47370,2265.78,9.07360
1,BTCUSDT,1775653725359956,72649.41,4.19763,72649.42,0.41097
2,BTCUSDT,1775653725360710,72649.41,4.19763,72649.42,0.40787
3,ETHUSDT,1775653725366797,2265.77,19.73440,2265.78,9.07360
4,BTCUSDT,1775653725371404,72649.41,4.19763,72649.42,0.33308


In [3]:
df["spread"] = (
    df["ask_price"] - df["bid_price"]
)

df["mid_price"] = (
    df["bid_price"] + df["ask_price"]
) / 2

df["spread_pct"] = (
    df["spread"] / df["mid_price"]
)

df["bid_notional"] = (
    df["bid_price"] * df["bid_qty"]
)

df["ask_notional"] = (
    df["ask_price"] * df["ask_qty"]
)

df["order_imbalance"] = (
    df["bid_notional"] - df["ask_notional"]
) / (
    df["bid_notional"] + df["ask_notional"]
)

df.head()

,symbol,event_time,bid_price,bid_qty,ask_price,ask_qty,spread,mid_price,spread_pct,bid_notional,ask_notional,order_imbalance
0,ETHUSDT,1775653725357699,2265.77,19.47370,2265.78,9.07360,0.01,2265.775,4.413501e-06,44122.925249,20558.781408,0.364309
1,BTCUSDT,1775653725359956,72649.41,4.19763,72649.42,0.41097,0.01,72649.415,1.376474e-07,304955.342898,29856.732137,0.821651
2,BTCUSDT,1775653725360710,72649.41,4.19763,72649.42,0.40787,0.01,72649.415,1.376474e-07,304955.342898,29631.518935,0.822877
3,ETHUSDT,1775653725366797,2265.77,19.73440,2265.78,9.07360,0.01,2265.775,4.413501e-06,44713.611488,20558.781408,0.370062
4,BTCUSDT,1775653725371404,72649.41,4.19763,72649.42,0.33308,0.01,72649.415,1.376474e-07,304955.342898,24198.068814,0.852968


In [4]:
df["microprice"] = (
    (
        df["bid_price"] * df["ask_qty"]
    ) +
    (
        df["ask_price"] * df["bid_qty"]
    )
) / (
    df["bid_qty"] + df["ask_qty"]
)

df["microprice_diff"] = (
    df["microprice"] - df["mid_price"]
)

In [5]:
df["return"] = (
    df.groupby("symbol")["mid_price"]
    .pct_change()
)

df["volatility_20"] = (
    df.groupby("symbol")["return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)

df["imbalance_change"] = (
    df.groupby("symbol")["order_imbalance"]
    .diff()
)

In [6]:
df["future_return_250"] = (
    df.groupby("symbol")["mid_price"]
    .shift(-250) / df["mid_price"]
) - 1

In [7]:
threshold = 0.001

df["big_move_direction"] = "SAME"

df.loc[
    df["future_return_250"] > threshold,
    "big_move_direction"
] = "UP"

df.loc[
    df["future_return_250"] < -threshold,
    "big_move_direction"
] = "DOWN"

In [8]:
big_moves_df = df[
    df["big_move_direction"] != "SAME"
].copy()

big_moves_df["big_move_direction"].value_counts()

big_move_direction
DOWN    55794
UP      45034
Name: count, dtype: int64

In [9]:
features = [
    "spread_pct",
    "order_imbalance",
    "imbalance_change",
    "microprice_diff",
    "volatility_20",
    "bid_notional",
    "ask_notional"
]

X = big_moves_df[features]

y = big_moves_df["big_move_direction"]

mask = X.notna().all(axis=1) & y.notna()

X = X[mask]
y = y[mask]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [11]:
predictions = model.predict(X_test)

print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

        DOWN       0.83      0.63      0.71     10104
          UP       0.70      0.87      0.78     10062

    accuracy                           0.75     20166
   macro avg       0.76      0.75      0.75     20166
weighted avg       0.76      0.75      0.74     20166



In [12]:
test_df = big_moves_df.loc[X_test.index].copy()

test_df = test_df.reset_index(drop=True)

test_df["predicted_direction"] = predictions

In [13]:
test_df["strategy_return"] = 0.0

test_df.loc[
    test_df["predicted_direction"] == "UP",
    "strategy_return"
] = test_df["future_return_250"]

test_df.loc[
    test_df["predicted_direction"] == "DOWN",
    "strategy_return"
] = -test_df["future_return_250"]

In [14]:
fee = 0.0004

test_df["net_return"] = (
    test_df["strategy_return"] - fee
)

test_df["net_return"].mean()

0.00019026911976598743